# Citation Index API — interactive client test (API 0.4.2)

Run this notebook from the project root with the package installed:

```bash
pip install -e ".[dev]"
pip install ipykernel   # if needed
python -m ipykernel install --user --name citation-index-api-client
```

Since API 0.4 every endpoint needs an API key (sent as `X-API-Key`). Set
`CITATION_INDEX_API_KEY` before starting Jupyter, or paste it into `API_KEY` below.

Select that kernel in Jupyter / VS Code, then run the cells top to bottom.

In [1]:
import os
from pathlib import Path

from citation_index_api_client import CitationIndexClient, __version__
from citation_index_api_client.exceptions import APIError, JobFailedError, JobTimeoutError

# --- configure ---
BASE_URL = "https://citation-index-api-graphia-app1-staging.apps.bst2.paas.psnc.pl"
API_KEY = os.environ.get("CITATION_INDEX_API_KEY")  # or paste a key here (don't commit it)
PDF_PATH = Path("paper.pdf")  # set to your PDF, or None to skip text extraction
POLL_INTERVAL = 2.0
JOB_TIMEOUT = 600.0

if not API_KEY:
    print("⚠ No API key — set CITATION_INDEX_API_KEY; every endpoint returns 401 without it")

client = CitationIndexClient(base_url=BASE_URL, api_key=API_KEY, timeout=120.0)
print(f"Client {__version__} → {client.base_url}")

Client 0.4.2 → https://citation-index-api-graphia-app1-staging.apps.bst2.paas.psnc.pl


## Health & API info

In [2]:
try:
    print("health:", client.health())
    print("root:", client.root())
except APIError as e:
    print(f"API error {e.status_code}: {e.detail}")

health: {'status': 'healthy', 'redis': 'ok', 'storage': 'ok', 'version': '0.4.2'}
root: {'name': 'Citation Index API', 'version': '0.4.2', 'docs': '/docs', 'health': '/health'}


## Extract text from a PDF

Set `PDF_PATH` in the first cell to a local `.pdf` file. This enqueues an async job and waits for the result.

In [3]:
PDF_PATH = Path("../test_data/reg_0035-2039_1989_num_102_485_2445.pdf")

In [4]:
if PDF_PATH is None or not PDF_PATH.exists():
    print(f"Skipping text extraction — set PDF_PATH to an existing file (got {PDF_PATH!r})")
else:
    job = client.extract_text(PDF_PATH, extractor="pymupdf", markdown=True)
    #job = client.extract_text(PDF_PATH, extractor="mineru", markdown=True)
    print(f"Enqueued: {job}")

    status = client.wait_for_job(
        job.job_id, 
        poll_interval=POLL_INTERVAL,
        timeout=JOB_TIMEOUT,
    )
    print(f"Finished: {status}")

    text_result = client.get_job_result(job.job_id)
    if isinstance(text_result, dict):
        preview = str(text_result)[:500]
    else:
        preview = str(text_result)[:500]
    print(f"Result preview:\n{preview}...")

Enqueued: JobResponse(job_id='d34265ec-d312-43a3-9a85-f3f3d525a10c', status='queued', created_at='2026-11-04T13:01:13.358677', message='Text extraction job enqueued')
Finished: JobStatus(job_id='d34265ec-d312-43a3-9a85-f3f3d525a10c', status='completed', created_at='2026-11-04T13:01:13.358677', started_at='2026-11-04T13:01:13.413586', completed_at='2026-11-04T13:01:20.919844', current_stage='text_extraction', completed_stages=['text_extraction'], error=None, progress=None)
Result preview:
{'text': "Revue des Études Grecques \n\n\n\nSur le troisième stasimon d'Œdipe-Roi Albert Machin \n\nCiter ce document / Cite this document : \n\nMachin Albert. Sur le troisième stasimon d'Œdipe-Roi. In: Revue des Études Grecques, tome 102, fascicule 485-486, Janvierjuin 1989. pp. 192-201 ; \n\ndoi : https://doi.org/10.3406/reg.1989.2445 \n\nhttps://www.persee.fr/doc/reg_0035-2039_1989_num_102_485_2445 \n\nFichier pdf généré le 18/04/2018 \n\n\n\n# SUR LE TROISIEME STASIMON O'ŒDIPE-ROI \n\n\n\n{O...


In [5]:
print(text_result['text'][:500])

Revue des Études Grecques 



Sur le troisième stasimon d'Œdipe-Roi Albert Machin 

Citer ce document / Cite this document : 

Machin Albert. Sur le troisième stasimon d'Œdipe-Roi. In: Revue des Études Grecques, tome 102, fascicule 485-486, Janvierjuin 1989. pp. 192-201 ; 

doi : https://doi.org/10.3406/reg.1989.2445 

https://www.persee.fr/doc/reg_0035-2039_1989_num_102_485_2445 

Fichier pdf généré le 18/04/2018 



# SUR LE TROISIEME STASIMON O'ŒDIPE-ROI 



{O.R. vientLe de 1086-1 chant prof


## Extract references from markdown

Uses a small sample bibliography block. Replace `SAMPLE_MARKDOWN` with output from text extraction if you have it.

In [6]:
#with open("../test_data/Holanda-Brasliadailyinvention-1989.md", "r") as f:
#    text_result = {}
#    text_result['text'] = f.read()

In [7]:
job = client.extract_references(
    text_result['text'],
    method="full_text",
    temperature=0.3,
)
print(f"Enqueued: {job}")

try:
    refs_result = client.wait_for_result(
        job.job_id,
        poll_interval=POLL_INTERVAL,
        timeout=JOB_TIMEOUT,
    )
    refs_result
except JobFailedError as e:
    print(f"Job failed: {e}")
except JobTimeoutError as e:
    print(f"Timed out after {e.timeout}s")

Enqueued: JobResponse(job_id='9990eba5-0041-4eff-8d0a-2fbc719af25f', status='queued', created_at='2026-11-04T13:01:21.905633', message='Reference extraction job enqueued')


In [8]:
refs_result

{'references': ['C. M. Bowra, Sophoclean Tragedy, Oxford, 1944, p. 199.',
  'A. Lesky, Greek Tragedy, éd. angl. Londres, 1965, p. 113.',
  'R. C. Jebb éd. O.T. Cambridge, 1914, p. 143.',
  'J. C. Kamerbeek éd. O.T. Leyde, 1967, p. 208.',
  "R. W. B. Burton, The Chorus in Sophocles' Tragedies, Oxford, 1980, p. 169-70.",
  'H. D. F. Kitto, Greek Tragedy, 3e éd., Londres, 1970, p. 167.',
  'R. C. Jebb, op. cit., comm. à 1086-1109.',
  'W. Kranz, Stasimon, Berlin, 1933, p. 213.',
  'G. M. Kirkwood, A Study of Sophoclean Drama, New York, 1958, p. 199-200.',
  'J. C. Kamerbeek, op. cit., comm. à 1086-1109.',
  'T. B. L. Webster, An Introduction to Sophocles, Oxford, 1936, p. 105.',
  'M. A. Bayfield, éd. Antigone, 1968.',
  'G. Mûller, éd. Antigone, 1967.',
  'A. M. Dale, «Stasimon and Hyporchème», Eranos, 48, 1950, p. 14-20.',
  'A. M. Dale, The Lyric Metres in Greek Drama, 2e éd., Cambridge, 1968, p. 208-10.',
  'P. E. Easterling, éd. Trach., Cambridge, 1982, p. 104.',
  'Burton, op. cit.,

## Parse reference strings

Send raw bibliography lines to the parser endpoint.

In [9]:
#job = client.parse_references(refs_result['references'], parser="grobid", temperature=0.0)
#job = client.parse_references(refs_result['references'], parser="llm", temperature=0.0, prompt_name="prompts/reference_parsing.md")
#job = client.parse_references(refs_result['references'], parser="llm", temperature=0.0, prompt_name="prompts/reference_parsing_linkedbook.md")
job = client.parse_references(refs_result['references'], parser="llm", temperature=0.0)
print(f"Enqueued: {job}")

parsed_result = client.wait_for_result(
    job.job_id,
    poll_interval=POLL_INTERVAL,
    timeout=JOB_TIMEOUT,
)
parsed_result

Enqueued: JobResponse(job_id='9fadba6e-8fc0-4c81-be8e-6c5e0245e09a', status='queued', created_at='2026-11-04T13:02:16.033290', message='Reference parsing job enqueued')


{'references': [{'full_title': 'Sophoclean Tragedy',
   'journal_title': None,
   'authors': [{'first_name': 'C. M.',
     'middle_name': None,
     'surname': 'Bowra',
     'name_link': None,
     'role_name': None}],
   'editors': None,
   'publisher': 'Oxford',
   'translator': None,
   'publication_place': 'Oxford',
   'publication_year': 1944,
   'publication_date_raw': '1944',
   'identifiers': [],
   'ref_type': 'book',
   'raw': {},
   'volume': None,
   'issue': None,
   'pages': '199',
   'cited_range': '199',
   'footnote_number': None},
  {'full_title': 'Greek Tragedy',
   'journal_title': None,
   'authors': [{'first_name': 'A.',
     'middle_name': None,
     'surname': 'Lesky',
     'name_link': None,
     'role_name': None}],
   'editors': None,
   'publisher': None,
   'translator': None,
   'publication_place': 'Londres',
   'publication_year': 1965,
   'publication_date_raw': '1965',
   'identifiers': [],
   'ref_type': 'book',
   'raw': {},
   'volume': None,
   'is

In [10]:
len(parsed_result['references'])

65

## Link references to external indexes

`link_references` submits the **structured records** produced by the parser above
to `POST /link/references` (as of API 0.3.1 it no longer accepts raw strings),
which resolves each one against OpenAlex, Matilda, Wikidata and (since API 0.4)
OpenCitations. Pass a list to
submit a batch, or a single record for one reference.

Each target returns an `id` and a `doi`, both `None` when no confident match is
found — the service requires a DOI hit or a near-exact title match. Linking hits
live external APIs once per reference per target, so keep `LINKING_LIMIT` small.

In [11]:
from citation_index_api_client import is_linkable

LINKING_TARGETS = "all"   # or "openalex", "matilda", "wikidata", "opencitations", or "openalex,wikidata"
LINKING_LIMIT = 5         # bounded: each reference hits every target's live API

# Linking consumes the parsed records, and the endpoint rejects the whole batch
# if any record lacks a title, authors or year — so keep only the linkable ones.
linkable = [r for r in parsed_result['references'] if is_linkable(r)]
print(f"{len(linkable)} of {len(parsed_result['references'])} records are linkable")
records_to_link = linkable[:LINKING_LIMIT]

job = client.link_references(records_to_link, target=LINKING_TARGETS)
print(f"Enqueued: {job}")

linked = client.wait_for_links(
    job.job_id,
    poll_interval=POLL_INTERVAL,
    timeout=JOB_TIMEOUT,
)
print(f"Linked {len(linked)} references")

28 of 65 records are linkable
Enqueued: JobResponse(job_id='8c1b2e22-fdc7-42c4-af02-6cea03505cea', status='queued', created_at='2026-11-04T13:07:44.176353', message='Citation linking job enqueued')
Linked 5 references


In [12]:
for item in linked:
    title = item.reference.get("full_title") or item.reference
    print(f"\n{str(title)[:90]}")
    for target, link in item.links.items():
        mark = "✓" if link.is_linked else "·"
        print(f"  {mark} {target:<9} id={link.id or '—'}  doi={link.doi or '—'}")

hits = sum(link.is_linked for item in linked for link in item.links.values())
lookups = sum(len(item.links) for item in linked)
print(f"\n{hits} / {lookups} target lookups matched")


Sophoclean Tragedy
  · openalex  id=—  doi=—
  · matilda   id=—  doi=—
  ✓ wikidata  id=Q123702058  doi=—
  · opencitations id=—  doi=—

Greek Tragedy
  · openalex  id=—  doi=—
  · matilda   id=—  doi=—
  · wikidata  id=—  doi=—
  · opencitations id=—  doi=—

The Chorus in Sophocles' Tragedies
  · openalex  id=—  doi=—
  · matilda   id=—  doi=—
  ✓ wikidata  id=Q123744614  doi=—
  · opencitations id=—  doi=—

Greek Tragedy
  · openalex  id=—  doi=—
  · matilda   id=—  doi=—
  · wikidata  id=—  doi=—
  · opencitations id=—  doi=—

Stasimon
  · openalex  id=—  doi=—
  · matilda   id=—  doi=—
  · wikidata  id=—  doi=—
  · opencitations id=—  doi=—

2 / 20 target lookups matched


## GROBID pipeline: PDF → references (API 0.4)

`process_references_grobid` goes straight from a PDF to index-aligned raw
`references` and `parsed_references` in one job — no separate text extraction or
parsing. With `include_context=True` it also returns `contexts`: the citing
sentences of each reference, ready for the classifier below.

In [13]:
job = client.process_references_grobid(PDF_PATH, include_context=True)
print(f"Enqueued: {job}")

grobid_result = client.wait_for_result(
    job.job_id,
    poll_interval=POLL_INTERVAL,
    timeout=JOB_TIMEOUT,
)
print(f"{len(grobid_result['references'])} references, "
      f"{len(grobid_result['parsed_references'])} parsed, "
      f"{len(grobid_result.get('contexts', []))} citing contexts")
grobid_result['contexts'][:3]

Enqueued: JobResponse(job_id='5085569b-a670-4420-b682-6ab0c49b1152', status='queued', created_at='2026-11-04T13:09:20.656188', message='GROBID reference extraction job enqueued')
29 references, 29 parsed, 29 citing contexts


[[], [], []]

`/extract/references` also accepts a PDF upload, which uses GROBID.
`extract_references_from_pdf` can include parsed records and citing sentences too.

In [14]:
job = client.extract_references_from_pdf(PDF_PATH, include_parsed=True, include_context=False)
pdf_refs = client.wait_for_result(job.job_id, poll_interval=POLL_INTERVAL, timeout=JOB_TIMEOUT)
print(sorted(pdf_refs))
pdf_refs['references'][:5]

['count', 'method', 'parsed_references', 'references']


['C. M. Bowra, Sophoclean Tragedy, Oxford, 1944, p. 199.',
 'Par ex. A. Lesky, Greek Tragedy, éd. angl. Londres, 1965, p. 113.',
 'R. C. Jebb éd. O.T. Cambridge, 1914, p. 143. J. C. Kamerbeek éd. O.T. Leyde, 1967, p. 208.',
 "R. W. B. Burton, The Chorus in Sophocles' Tragedies, Oxford, 1980, p. 169-70.",
 'H. D. F. Kitto, Greek Tragedy, 3e éd., Londres, 1970, p. 167 : «lyrical instrument».']

## Classify citation intent (API 0.4)

`classify_citations` labels each citing sentence as **method**, **background** or
**result**. Each item is `{"section": ..., "citation": ...}` (section may be `""`);
GROBID `contexts` can be passed as-is. `mode` is `"WS"` (with sections), `"WoS"`
(without) or `"M"` (per item, default).

In [ ]:
CLASSIFY_LIMIT = 10  # keep small while exploring

job = client.classify_citations(grobid_result['contexts'][:CLASSIFY_LIMIT], mode="M")
print(f"Enqueued: {job}")

intents = client.wait_for_result(job.job_id, poll_interval=POLL_INTERVAL, timeout=JOB_TIMEOUT)
intents

## Manual job polling (optional)

Use this pattern when you want to inspect status between polls instead of blocking on `wait_for_result`.

In [17]:
# Re-use the last job_id from the cell above, or paste one here:
job_id = job.job_id

status = client.get_job_status(job_id)
print(status)
print("terminal?", status.is_terminal)

if status.status == "completed":
    result = client.get_job_result(job_id)
    print(result)

JobStatus(job_id='960e8113-ac89-4ace-80ca-07f8a3e9160d', status='completed', created_at='2026-11-04T13:09:25.212654', started_at=None, completed_at='2026-11-04T13:09:28.593377', current_stage='reference_extraction', completed_stages=['reference_extraction'], error=None, progress=None)
terminal? True
{'references': ['C. M. Bowra, Sophoclean Tragedy, Oxford, 1944, p. 199.', 'Par ex. A. Lesky, Greek Tragedy, éd. angl. Londres, 1965, p. 113.', 'R. C. Jebb éd. O.T. Cambridge, 1914, p. 143. J. C. Kamerbeek éd. O.T. Leyde, 1967, p. 208.', "R. W. B. Burton, The Chorus in Sophocles' Tragedies, Oxford, 1980, p. 169-70.", 'H. D. F. Kitto, Greek Tragedy, 3e éd., Londres, 1970, p. 167 : «lyrical instrument».', 'W. Kranz, Stasimon, Berlin, 1933, p. 213. G. M. Kirkwood, A Study of Sophoclean Drama, New York, 1958, p. 199-200.', 'J. C. Kamerbeek, op. cit., comm. à 1086-1109.', 'T. B. L. Webster, An Introduction to Sophocles, Oxford, 1936, p. 105. du choeur «pur instrument» : on comprend dès lors que d

In [18]:
client.close()
print("Client closed.")

Client closed.
